In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/My Drive/WESAD_Project/WESAD/S2"))

['S2_quest.csv', 'S2.pkl', 'S2_readme.txt', 'S2_E4_Data.zip', 'S2_respiban.txt']


In [3]:
# ===== LOAD DATA =====
import pickle
import numpy as np

file_path = "/content/drive/My Drive/WESAD_Project/WESAD/S2/S2.pkl"

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

# ===== EXTRACT SIGNALS =====
signals = data['signal']['wrist']
eda = signals['EDA']
bvp = signals['BVP']
labels = data['label']

# ===== NORMALIZE SIGNALS =====
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
eda = scaler.fit_transform(eda)
bvp = scaler.fit_transform(bvp)

# ===== CONVERT LABELS TO BINARY =====
y_all = np.where(labels == 2, 1, 0)

# ===== FEATURE EXTRACTION FUNCTION =====
def extract_features(signal):
    return [
        np.mean(signal),
        np.std(signal),
        np.min(signal),
        np.max(signal)
    ]

# ===== SLIDING WINDOW + CORRECT LABEL ALIGNMENT =====
window_size = 100
features = []
labels_new = []

label_len = len(labels)
eda_len = len(eda)

for i in range(0, eda_len - window_size, window_size):

    # Map index
    label_idx = int(i * label_len / eda_len)

    # Skip unwanted labels FIRST
    if labels[label_idx] not in [1, 2, 3]:
        continue

    # THEN extract windows
    eda_window = eda[i:i+window_size]
    bvp_window = bvp[i:i+window_size]

    feat = extract_features(eda_window) + extract_features(bvp_window)

    # Add BOTH together
    features.append(feat)
    labels_new.append(y_all[label_idx])

# ===== FINAL DATA =====
X = np.array(features)
y = np.array(labels_new)

print("X shape:", X.shape)
print("y shape:", y.shape)

# ===== TRAIN TEST SPLIT =====
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X shape: (86, 8)
y shape: (86,)


In [5]:
# ===== RANDOM FOREST MODEL =====

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Create model
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

# Train model
rf_model.fit(X_train, y_train)

# Make predictions
rf_pred = rf_model.predict(X_test)

# ===== EVALUATION =====
print("===== Random Forest Results =====")

accuracy = accuracy_score(y_test, rf_pred)
precision = precision_score(y_test, rf_pred)
recall = recall_score(y_test, rf_pred)
f1 = f1_score(y_test, rf_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# ===== CONFUSION MATRIX =====
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

===== Random Forest Results =====
Accuracy: 0.9444444444444444
Precision: 0.8888888888888888
Recall: 1.0
F1 Score: 0.9411764705882353

Confusion Matrix:
[[9 1]
 [0 8]]


In [6]:
# ===== XGBOOST MODEL =====

!pip install xgboost

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Create model
xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=1,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
xgb_pred = xgb_model.predict(X_test)

# ===== EVALUATION =====
print("===== XGBoost Results =====")

accuracy = accuracy_score(y_test, xgb_pred)
precision = precision_score(y_test, xgb_pred)
recall = recall_score(y_test, xgb_pred)
f1 = f1_score(y_test, xgb_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# ===== CONFUSION MATRIX =====
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

===== XGBoost Results =====
Accuracy: 0.9444444444444444
Precision: 0.8888888888888888
Recall: 1.0
F1 Score: 0.9411764705882353

Confusion Matrix:
[[9 1]
 [0 8]]


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:55:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [7]:
# ===== LOGISTIC REGRESSION =====

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

#  Create model
lr_model = LogisticRegression(max_iter=1000)

#  Train
lr_model.fit(X_train, y_train)

#  Predict
lr_pred = lr_model.predict(X_test)

# ===== EVALUATION =====
print("===== Logistic Regression Results =====")

print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall:", recall_score(y_test, lr_pred))
print("F1 Score:", f1_score(y_test, lr_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred))

===== Logistic Regression Results =====
Accuracy: 0.8888888888888888
Precision: 0.875
Recall: 0.875
F1 Score: 0.875

Confusion Matrix:
[[9 1]
 [1 7]]


In [8]:
# ===== SVM MODEL =====

from sklearn.svm import SVC

#  Create model
svm_model = SVC(kernel='rbf', class_weight='balanced')

#  Train
svm_model.fit(X_train, y_train)

#  Predict
svm_pred = svm_model.predict(X_test)

# ===== EVALUATION =====
print("===== SVM Results =====")

print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

===== SVM Results =====
Accuracy: 0.8888888888888888
Precision: 0.8
Recall: 1.0
F1 Score: 0.8888888888888888

Confusion Matrix:
[[8 2]
 [0 8]]


In [4]:
# ===== LSTM MODEL (FIXED CODE) =====

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# ===== RESHAPE DATA FOR LSTM =====

X_seq = X.reshape(X.shape[0], X.shape[1], 1)
y_seq = y

print("LSTM Input Shape:", X_seq.shape)

# ===== TRAIN TEST SPLIT =====
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq,
    y_seq,
    test_size=0.2,
    random_state=42
)

# ===== BUILD LSTM MODEL =====
model = Sequential()

model.add(
    LSTM(
        64,
        input_shape=(X_seq.shape[1], X_seq.shape[2])
    )
)

model.add(Dense(1, activation='sigmoid'))

# ===== COMPILE MODEL =====
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ===== TRAIN MODEL =====
history = model.fit(
    X_train_seq,
    y_train_seq,
    epochs=10,
    batch_size=16,
    validation_data=(X_test_seq, y_test_seq)
)

# ===== PREDICTIONS =====
y_pred_prob = model.predict(X_test_seq)

# Convert probabilities to 0 or 1
y_pred_lstm = (y_pred_prob > 0.5).astype(int)

# Flatten array
y_pred_lstm = y_pred_lstm.flatten()

# ===== EVALUATION =====
print("\n===== LSTM Results =====")

accuracy = accuracy_score(y_test_seq, y_pred_lstm)
precision = precision_score(y_test_seq, y_pred_lstm)
recall = recall_score(y_test_seq, y_pred_lstm)
f1 = f1_score(y_test_seq, y_pred_lstm)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# ===== CONFUSION MATRIX =====
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_seq, y_pred_lstm))

LSTM Input Shape: (86, 8, 1)
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - accuracy: 0.6471 - loss: 0.6878 - val_accuracy: 0.7778 - val_loss: 0.6544
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9118 - loss: 0.6493 - val_accuracy: 0.8889 - val_loss: 0.6185
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9412 - loss: 0.6040 - val_accuracy: 0.8889 - val_loss: 0.5748
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9559 - loss: 0.5437 - val_accuracy: 0.8889 - val_loss: 0.5191
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9412 - loss: 0.4688 - val_accuracy: 0.8889 - val_loss: 0.4475
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9412 - loss: 0.3688 - val_accuracy: 0.8889 - val_loss: 0.3657
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9412 - loss: 0.2656 - val_accuracy: 0.8889 - val_loss: 0.3018
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9412 - loss: 0.2051 - val_accuracy: 0.8889 - val_loss: 0.2742
Epoch 9/10
